# Step 1: Opening the basic API

In [ ]:
from fastapi import FastAPI
import uvicorn

# 1. Initialize the FastAPI app
app = FastAPI(
    title="Local RAG API", 
    description="A fully local, private API to chat with your documents."
)

# 2. Create a simple "health check" route just to make sure it works
@app.get("/")
async def root():
    return {"message": "Welcome to the Local RAG API. The server is alive and running!"}

# --- We will add our /ingest and /ask routes here in the next steps ---

# 3. The command to start the server
# Every time Python runs a file, it secretly creates a few special background variables. One of those variables is called __name__.
    # Python uses this variable to keep track of how the file was opened:
        # Scenario A (Running it directly): If you run the file directly from your terminal by typing python main.py, 
            # It secretly sets __name__ = "__main__". The if statement becomes True, and the code inside runs.
        # Scenario B (Importing it): Imagine later you create a second file called testing.py, and inside it you write import main. 
            # Python says, "Ah, they are just borrowing code from main.py." It sets __name__ = "main". 
            # The if statement becomes False, and the code inside is skipped.
if __name__ == "__main__":
    print("Starting the FastAPI server on port 8000...")
    # FastAPI is just the framework we used to write our web routes. 
        # It doesn't actually know how to listen to the internet. 
        # For that, it needs a web server engine. That is what Uvicorn does.
    # When the if statement is True, it triggers Uvicorn:
        # app: This tells Uvicorn exactly which FastAPI application to run (the one we created at the top of the file).
        # host="0.0.0.0": This tells the server to listen to your "local" machine.
        # port=8000: Your computer has thousands of "ports" (virtual doors) that data can flow through. 
            # Web browsers usually use port 80 or 443. We are telling our API to listen specifically at door number 8000 so it doesn't interfere with anything else on your computer.
    uvicorn.run(app, host="0.0.0.0", port=8000)

Starting the FastAPI server on port 8000...


RuntimeError: asyncio.run() cannot be called from a running event loop

# Step 2: Creating the Ingestion Endpoint

In [ ]:
import os
import shutil
from fastapi import FastAPI, File, UploadFile
import uvicorn

# Import our LangChain tools
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

app = FastAPI(title="Local RAG API")

# --- GLOBAL SETUP ---
# We initialize our embedding model and database path out here 
# so they don't have to reboot every single time someone uploads a file.
print("Initializing Embeddings (Ensure Ollama is running!)...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
persist_directory = "./chroma_db"

@app.get("/")
async def root():
    return {"message": "Welcome to the Local RAG API. The server is alive!"}

# --- NEW: THE INGEST ROUTE ---
@app.post("/ingest")
# UploadFile is a special FastAPI class, it receives it as a safe, manageable "stream" of data.
# The File(...) part just tells FastAPI, mandates that this path has to receive a file, and it should be treated as such.
async def ingest_file(file: UploadFile = File(...)):
    print(f"Received file: {file.filename}")
    
    # 1. Save the uploaded file temporarily to our workspace
    temp_file_path = f"temp_{file.filename}"
    
    # When you upload that PDF, the data streaming across the network is raw, pure Binary Data (1s and 0s).
    # PyPDFLoader doesn't know how to read a network stream. It needs a physical file sitting on your computer's hard drive.
        # We create a temporary name like "temp_{file.filename}"
        # shutil.copyfileobj takes the network stream coming from the user and physically writes it to your 
        # VS Code workspace folder so LangChain can see it.
    
    # STREAMING vs WRITING TO PHYSICAL FILE
        # Streaming: When we say FastAPI receives it as a "stream", it means FastAPI is catching those packets 
        #   one by one and holding them in your computer’s RAM (faster)
        # Writing to Physical File: When I say we "write it to a VS Code folder," I mean we are taking that stream of data 
        #   out of the temporary RAM and saving it as a permanent, physical file on your Hard Drive (or SSD, which is slower).
    with open(temp_file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    
    # Now that the PDF is physically on your hard drive, we point LangChain at it. 
        # Why we can't point langchain directly at the pdf?
            # 1. When you read a .txt file, the computer reads it top-to-bottom, but for a pdf it is a complex database of fonts, images, and text boxes.
            # to know where Page 1 is, the computer actually has to look at a "map" (called the Cross-Reference Table), but this table located at the end
            # By writing it to the hard drive first, once the whole file is physically sitting on the drive, 
            # LangChain can jump straight to the end, read the map, and then jump back to the beginning to read the text.
            # 2. PyPDFLoader, is literally programmed to accept a File Path (a string of text like "C:/my_folder/document.pdf"). 
            # If you try to hand it a live, bubbling stream of binary network data, the Python code will instantly crash
    # .load() reads every single page of the PDF and converts it into a list of standardized Document objects.
    try:
        # 2. Load the PDF (Using PyPDFLoader instead of TextLoader)
        loader = PyPDFLoader(temp_file_path)
        docs = loader.load()
        # 3. Chunk it up
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(docs)
        # 4. Embed and save to ChromaDB
        vector_db = Chroma.from_documents(
            documents=chunks, 
            embedding=embeddings, 
            persist_directory=persist_directory
        )
        # 5. Clean up the temporary file so we don't clog our hard drive
        os.remove(temp_file_path)
        return {
            "message": f"Successfully processed {file.filename}!",
            "chunks_saved": len(chunks)
        }
    except Exception as e:
        # If anything goes wrong, still try to delete the temp file, then report the error
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
        return {"error": str(e)}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Step 3: Ask Endpoint

In [ ]:
import os
import shutil
from fastapi import FastAPI, File, UploadFile
import uvicorn
from pydantic import BaseModel

# Import our LangChain tools
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from sentence_transformers import CrossEncoder

app = FastAPI(title="Local RAG API")

# --- GLOBAL SETUP ---
print("Waking up the AI models... (This takes a few seconds)")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
reranker = CrossEncoder("BAAI/bge-reranker-base")
llm = Ollama(model="llama3.1") # Make sure you have this model pulled in Ollama!

persist_directory = "./chroma_db"

@app.get("/")
async def root():
    return {"message": "Welcome to the Local RAG API. The server is alive!"}

# --- 1. THE INGEST ROUTE ---
@app.post("/ingest")
async def ingest_file(file: UploadFile = File(...)):
    print(f"Received file: {file.filename}")
    temp_file_path = f"temp_{file.filename}"
    
    with open(temp_file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
        
    try:
        loader = PyPDFLoader(temp_file_path)
        docs = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(docs)
        
        vector_db = Chroma.from_documents(
            documents=chunks, embedding=embeddings, persist_directory=persist_directory
        )
        
        os.remove(temp_file_path)
        return {"message": f"Successfully processed {file.filename}!", "chunks_saved": len(chunks)}
        
    except Exception as e:
        if os.path.exists(temp_file_path): os.remove(temp_file_path)
        return {"error": str(e)}

# --- NEW: PYDANTIC BLUEPRINT ---
# This tells FastAPI that the /ask door expects a JSON object with a "query" string
class AskRequest(BaseModel):
    query: str

# --- NEW: THE ASK ROUTE ---
@app.post("/ask")
async def ask_question(request: AskRequest):
    query = request.query
    print(f"\nUser asked: '{query}'")
    
    # 1. Connect to our saved database
    vector_db = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    
    # 2. Stage 1: Broad Retrieval (The Intern)
    broad_results = vector_db.similarity_search(query, k=10)
    if not broad_results:
        return {"answer": "My database is empty! Please upload a PDF to /ingest first."}
        
    # 3. Stage 2: Reranking (The Expert)
    pairs = [[query, doc.page_content] for doc in broad_results]
    scores = reranker.predict(pairs)
    scored_docs = zip(broad_results, scores)
    sorted_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    
    # Get the text of the top 3 absolute best chunks
    top_chunks = [doc.page_content for doc, score in sorted_docs[:3]]
    
    # Glue the chunks together into one big block of context text
    context_text = "\n\n---\n\n".join(top_chunks)
    
    # 4. Build the final prompt for Llama 3.1
    prompt = f"""You are a helpful, brilliant assistant. Use ONLY the provided context to answer the user's question. 
    If the answer is not in the context, do not guess. Just say "I cannot answer this based on the provided documents."
    
Context from Database:
{context_text}

User Question:
{query}

Your Answer:"""
    
    # 5. Generate the answer!
    print("Thinking...")
    answer = llm.invoke(prompt)
    print("Done!")
    
    # 6. Return the answer AND the sources we used to get it
    return {
        "question": query,
        "answer": answer,
        "sources_used": top_chunks
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)